# **Métricas y gráficas de los 3 OCRs para elección de uno**

Obtiene las métricas y gráficas de los 3 OCRs para la elección de uno para el sistema final desarrollado en el TFM.

Se siguen los siguientes pasos:

1. **Importación de librerías**: Se importan las librerías utilizadas.
2. **Obtención y tratamiento de predicciones de OCR**: Se guardan las predicciones de cada OCR en un DataFrame y se eliminan las filas cuyo número real de dorsal no está anotado (está vacío). Se tratan las columnas para tener únicamente un dígito y una confianza por imagen para cada OCR.
5. **Métricas por umbral de confianza**: Se calculan las métricas para cada OCR para diferentes umbrales de confianza y se crean las gráficas.
6. **Métricas por dorsal**: Se calculan las métricas por dorsal.

## **Importación de librerías**

In [ ]:
import pandas as pd
import numpy as np

## **Obtención y tratamiento de predicciones de OCR**

### **Obtención de predicciones**

Se obtienen los datos del CSV en el que se almacenaron y se guardan en un DataFrame. Si el número no está anotado, se elimina la fila.

In [ ]:
df_resultados = pd.read_csv(f"./resultados_ocr_todo.csv")
print(f"Total filas: {len(df_resultados)}")

# Si numero_real está vacío, eliminar la fila
df_resultados = df_resultados[df_resultados["numero_real"].notna()]
print(f"Filas después de eliminar vacíos: {len(df_resultados)}")
df_resultados

### **Tratamiento de predicciones**

Una vez obtenidas las predicciones, se limpian las columnas del DataFrame para tener como texto un solo str y confianza un solo float. Si str tiene caracter no numérico, str=""

In [ ]:
df = df_resultados.copy()
import ast
# Para que se de cuenta que es una lista
df["tesseract_confianza"] = df["tesseract_confianza"].apply(ast.literal_eval)
df["tesseract_texto"] = df["tesseract_texto"].apply(ast.literal_eval)

#### **Tesseract**

Se queda con la cadena de solo dígitos que tenga mayor confianza.

In [ ]:
def limpiar_tesseract_mejor_digito(texts, confs):
    if texts is None or confs is None:
        return "", np.nan

    candidatos = []
    
    for t, c in zip(texts, confs):
        t_str = "" if t is None else str(t)
        t_str = t_str.strip()
        if t_str == "":
            continue

        # Mirar cada carácter del texto, si es un dígito
        digito = True
        for char in t_str:
            if not char.isdigit():
                digito = False
                break
        if digito:
            candidatos.append((t_str, float(c)/100.0))

    if not candidatos:
        return "", 0.0

    # Elegir el str (todos los caracteres son dígitos) con mayor confianza
    mejor_digito, mejor_conf = max(candidatos, key=lambda x: x[1])

    return mejor_digito, float(mejor_conf)

df["tesseract_texto"], df["tesseract_confianza"] = zip(*df.apply(lambda row: limpiar_tesseract_mejor_digito(row["tesseract_texto"], row["tesseract_confianza"]), axis=1))

#### **EasyOCR**

Se crea una función para limpiar la lista de confianza de EasyOCR para obtener una lista de números en lugar de una cadena de texto. Después, se queda con la cadena de solo dígitos que tenga mayor confianza. Por último, pasa la confianza de la cadena final a la escala [0, 100] (dado que estaba en la escala [0, 1]).

In [ ]:
def parse_npfloat_list_manual(s):
    s = str(s).strip()
    if s.startswith("[") and s.endswith("]"):
        s = s[1:-1]  # quitar corchetes

    # Quitar 'np.float64(' y ')'
    s = s.replace("np.float64(", "").replace(")", "")
    # Ahora s es algo como "0.0012139599863752338, 0.28118570217780364"
    
    parts = [p.strip() for p in s.split(",") if p.strip() != ""]

    floats = []
    for p in parts:
        try:
            floats.append(float(p))
        except ValueError:
            continue

    return floats

df["easyocr_confianza"] = df["easyocr_confianza"].apply(parse_npfloat_list_manual)
df["easyocr_texto"] = df["easyocr_texto"].apply(ast.literal_eval)

df["easyocr_texto"], df["easyocr_confianza"] = zip(*df.apply(lambda row: limpiar_tesseract_mejor_digito(row["easyocr_texto"], row["easyocr_confianza"]), axis=1))
# easyocr_confianza * 100 porque originalmente era un porcentaje en [0,1]
df["easyocr_confianza"] = df["easyocr_confianza"].apply(lambda x: x * 100 if not pd.isna(x) else np.nan)

df

#### **PARSeq**

Dado que PARSeq da confianza por cada caracter de la cadena, nos quedamos con la confianza mínima.

In [ ]:
def extraer_numeros_tensor(cadena):
    # Si por algún motivo ya es una lista o un valor nulo, lo devolvemos tal cual
    if not isinstance(cadena, str):
        return cadena
        
    try:
        # Extraemos solo lo que hay entre los corchetes [ y ]
        contenido = cadena.split('[')[1].split(']')[0]
        
        # Separamos por comas, limpiamos espacios y convertimos a número (float)
        lista_numeros = [float(numero.strip()) for numero in contenido.split(',')]
        
        return lista_numeros
    except IndexError:
        # Por si alguna cadena no tiene el formato esperado
        return []

df["parseq_confianza"] = df["parseq_confianza"].apply(extraer_numeros_tensor)
# Quedarse con cnfianza mínima
df["parseq_confianza"] = df["parseq_confianza"].apply(lambda lst: min(lst) if isinstance(lst, list) and len(lst) > 0 else np.nan)

In [ ]:
df

#### **Para los 3 OCRs**

Para cada OCR, si el texto no es un dígito, poner confianza 0 y texto vacío

In [ ]:
# Para cada OCR, si el texto no es un dígito, poner confianza 0 y texto vacío
for ocr in ["tesseract", "easyocr", "parseq"]:
    columna_texto = f"{ocr}_texto"
    columna_confianza = f"{ocr}_confianza"

    df[columna_texto] = df[columna_texto].astype(str)

    mascara_no_digitos = ~df[columna_texto].str.match(r"^\s*\d+\s*$", na=False)

    df.loc[mascara_no_digitos, columna_texto] = ""
    df.loc[mascara_no_digitos, columna_confianza] = 0.0

## **Métricas por umbral de confianza**

### **Función para obtener las métricas**

Se crea una función para calcular métricas de cada OCR

In [ ]:
import numpy as np

def compute_metrics_for_ocr(df, ocr_name, split=None, min_conf=None):
    """
    Función para calcular las métricas de un OCR (ocr_name) en un split o varios (puede ser str o List) y dado un umbral de confianza mínimo (min_conf).
    """
    columna_texto = f"{ocr_name}_texto"
    columna_confianza = f"{ocr_name}_confianza"

    data = df.copy()

    # Sólo los splits indicados
    if split is not None:
        if isinstance(split, str):
            data = data[data["split"] == split]
        elif isinstance(split, list):
            data = data[data["split"].isin(split)]

    # Si no hay imágenes tras filtrar, se devuelven las métricas a 0
    total = len(data)
    if total == 0:
        return {
            "ocr": ocr_name,
            "split": split,
            "min_conf": min_conf,
            "total": 0,
            "n_aceptadas": 0,
            "n_aceptadas_correctas": 0,
            "n_aceptadas_erroneas": 0,
            "porcentaje_aceptadas": 0.0,
            "porcentaje_correctas": 0.0,
            "porcentaje_descartadas": 1.0,
        }

    # Obtener filas con confianza mínima
    mascara_min_conf = True
    if min_conf is not None:
        mascara_min_conf = data[columna_confianza] >= min_conf

    # Imágenes aceptadas = con confianza suficiente y texto no vacío
    texto = data[columna_texto].astype(str)
    mascara_txt_no_vacio = ~(texto.isna() | (texto.str.len() == 0))
    mascara_aceptadas = mascara_min_conf & mascara_txt_no_vacio
    data_aceptadas = data[mascara_aceptadas]
    n_aceptadas = len(data_aceptadas)

    # Imágenes correctas de las aceptadas
    texto_aceptado_predicho = data_aceptadas[columna_texto].astype(str)
    texto_aceptado_gt = data_aceptadas["numero_real"].astype(int).astype(str)
    n_aceptadas_correctas = (texto_aceptado_gt == texto_aceptado_predicho).sum()
    n_aceptadas_erroneas = n_aceptadas - n_aceptadas_correctas

    # Porcentaje de imágenes aceptadas entre las totales
    porcentaje_aceptadas = n_aceptadas / total

    # Porcentaje de imágenes correctas entre las aceptadas
    porcentaje_correctas = (
        n_aceptadas_correctas / n_aceptadas if n_aceptadas > 0 else np.nan
    )

    # Porcentaje de imágenes descartadas sobre el total
    porcentaje_descartadas = 1 - (n_aceptadas / total)

    return {
        "ocr": ocr_name,
        "split": split,
        "min_conf": min_conf,
        "total": int(total),
        "n_aceptadas": int(n_aceptadas),
        "n_aceptadas_correctas": int(n_aceptadas_correctas),
        "n_aceptadas_erroneas": int(n_aceptadas_erroneas),
        "porcentaje_aceptadas": float(porcentaje_aceptadas),
        "porcentaje_correctas": float(porcentaje_correctas) if not np.isnan(porcentaje_correctas) else np.nan,
        "porcentaje_descartadas": float(porcentaje_descartadas),
    }

### **Cálculo de métricas para cada OCR y split (confianza 0)**

Se calculan las métricas para los tres OCRs para los tres splits con confianza mínima de 0.

In [ ]:
import numpy as np
ocrs = ["tesseract", "easyocr", "parseq"]
splits = ["train", "valid", "test"]

resultados = []
for ocr in ocrs:
    for s in splits:
        m = compute_metrics_for_ocr(df, ocr_name=ocr, split=s, min_conf=0.0)
        resultados.append(m)

metricas_df = pd.DataFrame(resultados)
metricas_df

### **Cálculo de métricas para cada OCR y umbral de confianza**

Se calculan las métricas para diferentes umbrales de confianza

In [ ]:
thresholds = [x / 100 for x in range(0, 101, 5)]
resultados_thr = []

for ocr in ocrs:
    for t in thresholds:
        m = compute_metrics_for_ocr(df, ocr_name=ocr, split=["train", "valid", "test"], min_conf=t)
        resultados_thr.append(m)

metricas_thr_df = pd.DataFrame(resultados_thr)
metricas_thr_df

### **Creación de gráficas para cada OCR**

Se crea una función para dibujar las métricas calculadas

In [ ]:
import matplotlib.pyplot as plt

def dibujar_metricas(metricas_df, split=["train", "valid", "test"]):
    """
    Dibuja el porcentaje de imágenes aceptadas y porcentaje de imágenes correctas entre las aceptadas para cada umbral de confianza. Dibuja un plot por cada OCR.
    """
    # Splits
    if isinstance(split, str):
        df = metricas_df[metricas_df["split"] == split].copy()
    elif isinstance(split, list):
        df = metricas_df[metricas_df["split"].isin(split)].copy()
    else:
        df = metricas_df.copy()

    # OCRs
    ocrs = df["ocr"].unique()
    print(f"Métricas para split='{split}' con OCRs: {ocrs}")

    for ocr in ocrs:
        sub = df[df["ocr"] == ocr].copy()
        if sub.empty:
            continue

        # Ordenar por umbral de confianza
        sub = sub.sort_values("min_conf")

        plt.figure(figsize=(6, 4))

        plt.plot(
            sub["min_conf"],
            sub["porcentaje_correctas"],
            marker="o",
            label="Lecturas correctas entre las consideradas",
        )

        plt.plot(
            sub["min_conf"],
            sub["porcentaje_aceptadas"],
            marker="s",
            label="Imágenes consideradas",
        )

        plt.xlabel("")
        plt.ylabel("")
        plt.ylim(0, 1.05)
        plt.title(f"")
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

Se crean las gráficas con la función anteriormente creada

In [ ]:
dibujar_metricas(metricas_thr_df, split=None)

## **Métricas por dorsal**

Se crea la función para sacar las métricas por dorsal

In [ ]:
def metricas_por_dorsal(df, ocr_name, split=None):
    columna_texto = f"{ocr_name}_texto"

    data = df.copy()
    if split is not None:
        if isinstance(split, str):
            data = data[data["split"] == split]
        elif isinstance(split, list):
            data = data[data["split"].isin(split)]

    # Definir máscaras de vacío / no vacío
    texto = data[columna_texto].astype(str)
    mascara_blanco = texto.isna() | (texto.str.len() == 0)
    mascara_no_blanco = ~mascara_blanco

    # Columna de acierto sólo en las que dicen algo
    gt = data["numero_real"].astype(int).astype(str)
    pred = texto
    es_correcto = (gt == pred) & mascara_no_blanco

    data = data.assign(
        es_blanco=mascara_blanco,
        es_correcto=es_correcto,
        es_no_blanco=mascara_no_blanco,
    )

    resumen = (
        data
        .groupby("numero_real")
        .agg(
            n_total=("numero_real", "size"),
            n_blanco=("es_blanco", "sum"),
            n_no_blanco=("es_no_blanco", "sum"),
            n_correctos=("es_correcto", "sum"),
        )
        .reset_index()
    )

    # Porcentajes
    resumen["porc_blanco"] = resumen["n_blanco"] / resumen["n_total"]
    resumen["porc_acierto"] = resumen["n_correctos"] / resumen["n_total"]
    resumen["porc_acierto_sobre_no_blanco"] = resumen["n_correctos"] / resumen["n_no_blanco"].replace(0, np.nan)

    return resumen.sort_values("porc_blanco", ascending=False)

Se calculan las métricas por dorsal de cada OCR y se muestra las de parseq

In [ ]:
metricas_dorsal_tesseract = metricas_por_dorsal(df, ocr_name="tesseract")
metricas_dorsal_easyocr = metricas_por_dorsal(df, ocr_name="easyocr")
metricas_dorsal_parseq = metricas_por_dorsal(df, ocr_name="parseq")
 
metricas_dorsal_parseq

Se muestran el mínimo y máximo del porcentaje de dorsales sobre los que no tiene respuesta el OCR y, sobre el porcentaje de dorsales que tiene respuesta, se muestra el porcentaje mínimo y máximo de aciertos.

In [ ]:
# Mínimo y máximo de porc_blanco para parseq
min_blanco_parseq = metricas_dorsal_parseq["porc_blanco"].min()
max_blanco_parseq = metricas_dorsal_parseq["porc_blanco"].max()
print(f"Parseq - porc_blanco mínimo: {min_blanco_parseq:.4f}, máximo: {max_blanco_parseq:.4f}")

# Mínimo y máximo de porc_acierto_sobre_no_blanco para parseq
min_acierto_sobre_no_blanco_parseq = metricas_dorsal_parseq["porc_acierto_sobre_no_blanco"].min()
max_acierto_sobre_no_blanco_parseq = metricas_dorsal_parseq["porc_acierto_sobre_no_blanco"].max()
print(f"Parseq - porc_acierto_sobre_no_blanco mínimo: {min_acierto_sobre_no_blanco_parseq:.4f}, máximo: {max_acierto_sobre_no_blanco_parseq:.4f}")